# Phase 2 — Multi-turn Chatbot, System Prompts, Temperature, Streaming

Phase 1 sent one message and got one reply. That's all the API does. To get a **conversation**, *you* have to remember the history and resend it every time.

## What you'll learn
1. **Multi-turn** — the API is stateless; you keep the message list
2. **System prompts** — give Claude a role/personality
3. **Temperature** — control randomness (0 = deterministic, 1 = creative)
4. **Streaming** — show text as it's generated instead of waiting

By the end you'll have a tiny chatbot loop you can talk to in the notebook.

## Setup (same as Phase 1)

In [ ]:
from dotenv import load_dotenv
load_dotenv(dotenv_path="../.env")

from anthropic import Anthropic
client = Anthropic()
model = "claude-sonnet-4-5"

## Step 1 — Helper functions

Three tiny helpers that we'll reuse for the rest of the course. The whole conversation lives in a plain Python list.

- `add_user_message(messages, text)` — append a user turn
- `add_assistant_message(messages, text)` — append an assistant turn
- `chat(messages, ...)` — send the whole history, return the reply text

In [ ]:
def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})

def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    response = client.messages.create(**params)
    return response.content[0].text

## Step 2 — Prove the API has no memory

First, the wrong way: two **separate** calls. The second call has no idea what the first one was about.

In [ ]:
# Call 1 — fresh
msgs_a = []
add_user_message(msgs_a, "My name is Giorgi.")
print("Claude:", chat(msgs_a))

# Call 2 — also fresh (NEW list, no history)
msgs_b = []
add_user_message(msgs_b, "What is my name?")
print("Claude:", chat(msgs_b))  # Claude has no idea

Now the right way: keep one list, append every turn.

In [ ]:
messages = []

add_user_message(messages, "My name is Giorgi.")
reply = chat(messages)
add_assistant_message(messages, reply)
print("Claude:", reply)

add_user_message(messages, "What is my name?")
reply = chat(messages)
add_assistant_message(messages, reply)
print("Claude:", reply)  # Should now know it's Giorgi

print("\nFinal message history:")
for m in messages:
    print(f"  [{m['role']}] {m['content'][:60]}...")

**Important:** the message list grows on every turn. That means every call gets more expensive (more input tokens). Caching (Phase 6) is the answer when this becomes a problem.

## Step 3 — System prompts give Claude a personality

A system prompt controls **how** Claude responds, not **what** it responds. Same question → different style depending on the role.

Let's compare two personalities answering the same math question.

In [ ]:
question = "What is 47 times 12?"

# Personality 1 — direct answerer
msgs = []
add_user_message(msgs, question)
print("DIRECT:")
print(chat(msgs, system="Answer concisely. Give just the number."))

print("\n---\n")

# Personality 2 — Socratic tutor
msgs = []
add_user_message(msgs, question)
print("TUTOR:")
print(chat(msgs, system=(
    "You are a patient math tutor. Never give the answer directly. "
    "Instead, ask the student a guiding question that helps them work it out themselves."
)))

Same model, same input, completely different output. That's the whole power of system prompts.

## Step 4 — Temperature: deterministic vs creative

- `temperature=0` → always picks the most likely next token → near-deterministic
- `temperature=1` → picks lower-probability tokens more often → more varied/creative

Rule of thumb: low for extraction/factual, high for creative writing.

Let's run the same creative prompt twice at each temperature and compare.

In [ ]:
prompt = "Write a one-sentence opening for a noir detective novel."

for temp in (0.0, 1.0):
    print(f"--- temperature={temp} ---")
    for i in range(2):
        msgs = []
        add_user_message(msgs, prompt)
        print(f"  run {i+1}:", chat(msgs, temperature=temp))
    print()

At `0.0` the two runs should be nearly identical. At `1.0` they should be visibly different.

## Step 5 — Streaming: show text as it arrives

Without streaming, you wait 5–30 seconds, then the whole reply lands at once. With streaming, characters print as they're generated — same total time, much better UX.

Easiest API: `client.messages.stream()` as a context manager, then iterate `stream.text_stream`.

In [ ]:
with client.messages.stream(
    model=model,
    max_tokens=400,
    messages=[{"role": "user", "content": "Tell me a short story about a stubborn teapot. Keep it under 150 words."}],
) as stream:
    for chunk in stream.text_stream:
        print(chunk, end="", flush=True)

    # After the loop, you can grab the full message (useful for saving to history)
    final = stream.get_final_message()

print("\n\nStop reason:", final.stop_reason)
print("Output tokens:", final.usage.output_tokens)

## Step 6 — Tie it all together: a tiny chatbot loop

Run this cell, then type messages in the prompt that appears. Type `exit` to stop.

It uses everything from this phase: persistent message history, a system prompt, streaming output.

In [ ]:
system_prompt = (
    "You are a friendly programming mentor. "
    "Keep replies short (3-4 sentences max) unless the user asks for detail. "
    "Use plain language, no jargon dumps."
)

messages = []

while True:
    user_text = input("\nYou: ").strip()
    if user_text.lower() in {"exit", "quit", ""}:
        print("Bye!")
        break

    add_user_message(messages, user_text)

    print("Claude: ", end="", flush=True)
    with client.messages.stream(
        model=model,
        max_tokens=500,
        system=system_prompt,
        messages=messages,
    ) as stream:
        for chunk in stream.text_stream:
            print(chunk, end="", flush=True)
        final = stream.get_final_message()

    add_assistant_message(messages, final.content[0].text)
    print()  # newline after streamed reply

## Mini-exercise

1. **Personality swap.** Re-run the chatbot loop with a different `system_prompt` — try "You are a grumpy pirate who answers every question with at least one nautical metaphor." Notice how the *answers* are still correct but the *style* completely changes.
2. **Memory check.** In the chatbot, tell it your favorite color. Then ask 3 unrelated questions. Then ask "what was my favorite color?" — it should still remember, because every turn resends the full history.
3. **Token watch.** Print `len(messages)` and `final.usage.input_tokens` after each turn. Watch input_tokens grow as the conversation gets longer. This is the cost problem prompt caching solves later.

When you've done all three, ping me with one thing you noticed — anything surprising — and we'll go to **Phase 3: prefilling + stop sequences + clean structured JSON output**.